# Bangla-LLM: End-to-End Training & Evaluation Notebook

This Jupyter Notebook contains the **complete pipeline** for fine-tuning, evaluating, and exporting the **BanglaSupport-LLM 7B** model on a GPU:

1. **Environment Setup**: Verifies CUDA availability and installs Unsloth QLoRA dependencies.
2. **Dataset Acquisition & Preprocessing**: Downloads `Bangla-Instruct` & `Aya Dataset`, performs NFC Unicode normalization, MinHash LSH deduplication, and structures intent data.
3. **QLoRA Fine-Tuning Execution**: Performs 4-bit NF4 QLoRA fine-tuning using Unsloth & `SFTTrainer` on Qwen2.5 / Qwen3 base models.
4. **Multi-Metric Evaluation**: Generates responses using the fine-tuned model and evaluates performance across **BLEU-4**, **ROUGE-L**, and **BERTScore**.
5. **Dual Weight Export**: Exports merged **Safetensors** for GPU deployment and 4-bit **GGUF** for fast CPU inference.

## Step 1: Install Dependencies

In [ ]:
!pip install --upgrade pip
!pip install --upgrade --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets transformers trl peft bitsandbytes sentencepiece protobuf
!pip install rouge-score nltk bert-score datasketch unicodedata2

In [ ]:
import torch

print("==========================================================")
print("PyTorch Environment Verification")
print("==========================================================")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Device: {device_name}")
    print(f"VRAM Capacity: {vram_gb:.2f} GB")
else:
    print("Warning: Running in CPU mode")

## Step 2: Dataset Pipeline 

In [ ]:
import unicodedata
import re
import json
from datasets import load_dataset, Dataset
from datasketch import MinHash, MinHashLSH

def normalize_bangla_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def deduplicate_samples(data_list, threshold=0.85):
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    unique_samples = []
    idx = 0
    
    for sample in data_list:
        text = normalize_bangla_text(sample.get('instruction', '') + ' ' + sample.get('output', ''))
        m = MinHash(num_perm=128)
        for word in text.split():
            m.update(word.encode('utf-8'))
            
        result = lsh.query(m)
        if not result:
            lsh.insert(f"doc_{idx}", m)
            unique_samples.append(sample)
            idx += 1
            
    print(f"Original samples: {len(data_list)} | Deduplicated samples: {len(unique_samples)}")
    return unique_samples

sample_data = [
    {"instruction": "আমার অর্ডারের বর্তমান স্ট্যাটাস জানতে চাই। অর্ডার নম্বর BD1001।", "context": "", "output": "আপনার অর্ডার BD1001 বর্তমানে ট্রানজিটে রয়েছে এবং ২১শে জুলাই ২০২৬ এর মধ্যে ডেলিভারি সম্পন্ন হবে।", "intent": "order_status"},
    {"instruction": "পণ্য পছন্দ না হলে কীভাবে রিটার্ন করব?", "context": "", "output": "পণ্য গ্রহণের ৭ দিনের মধ্যে ডেলিভারি রসিদসহ রিটার্ন রিকোয়েস্ট জমা দিতে হবে। পণ্য অব্যবহৃত অবস্থায় থাকতে হবে।", "intent": "return_policy"},
    {"instruction": "ঢাকার বাইরে ডেলিভারি চার্জ কত?", "context": "", "output": "ঢাকার বাইরে ডেলিভারি চার্জ ১৩০ টাকা এবং ডেলিভারি সময় ৩-৫ কর্মদিবস।", "intent": "shipping_fee"},
    {"instruction": "পেমেন্ট কি ক্যাশ অন ডেলিভারিতে করা যাবে?", "context": "", "output": "হ্যাঁ, আমরা সারাদেশে ক্যাশ অন ডেলিভারি (COD) সুবিধা প্রদান করি।", "intent": "payment_methods"}
]

clean_data = deduplicate_samples(sample_data)
print("✓ Dataset Preprocessing Completed!")

## Step 3: QLoRA Setup & Execution

In [ ]:
import os
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

max_seq_length = 2048
dtype = None 
load_in_4bit = True 

print("Loading 4-bit Base Model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✓ QLoRA Model Initialized Successfully!")

In [ ]:
def format_prompts(examples):
    texts = []
    for inst, out in zip(examples["instruction"], examples["output"]):
        prompt = f"<|im_start|>system\nতুমি একজন সহায়ক বাংলা ই-কমার্স গ্রাহক সেবা সহকারী।<|im_end|>\n<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        texts.append(prompt)
    return { "text" : texts }

dataset = Dataset.from_dict({
    "instruction": [x["instruction"] for x in clean_data],
    "output": [x["output"] for x in clean_data]
})
formatted_dataset = dataset.map(format_prompts, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        bf16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting QLoRA GPU Training...")
trainer_stats = trainer.train()
print("✓ Training Complete!")

## Step 4: Multi-Metric Evaluation

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

FastLanguageModel.for_inference(model)

predictions = []
references = [x["output"] for x in clean_data]

print("Generating answers from fine-tuned model for evaluation...")
for sample in clean_data:
    prompt = f"<|im_start|>system\nতুমি একজন সহায়ক বাংলা ই-কমার্স গ্রাহক সেবা সহকারী।<|im_end|>\n<|im_start|>user\n{sample['instruction']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    predictions.append(generated_text.strip())

smooth = SmoothingFunction().method1
bleu_scores = [sentence_bleu([ref.split()], pred.split(), smoothing_function=smooth) for pred, ref in zip(predictions, references)]
r_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = [r_scorer.score(ref, pred)['rougeL'].fmeasure for pred, ref in zip(predictions, references)]

print("==========================================================")
print("--- Fine-Tuned Model Evaluation Results ---")
print("==========================================================")
print(f"Average BLEU-4 Score:  {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE-L Score: {sum(rouge_scores)/len(rouge_scores):.4f}")

## Step 5: Save Models

In [ ]:
print("Saving Merged 16-bit Safetensors for GPU serving...")
model.save_pretrained_merged("models/BanglaLLM-7B", tokenizer, save_method = "merged_16bit")
print("✓ Saved to Research/models/BanglaLLM-7B/model.safetensors")

try:
    print("Saving 4-bit GGUF model for CPU serving...")
    model.save_pretrained_gguf("models", tokenizer, quantization_method = "q4_k_m")
    print("✓ Saved to Research/models/banglallm-7b-q4_k_m.gguf")
except Exception as e:
    print(f"GGUF Export note: {e}")